# ABO Blood Group Classification Pipeline

## Overview
This pipeline classifies ABO blood groups from Raman spectroscopy data using a
pairwise binary SVM approach, following Jensen et al. (2024). Each of the four 
blood groups (A, B, AB, O) is compared against every other group in a one-vs-one 
scheme, resulting in six binary classifiers.

---

## Step 1: Data Loading
Raw Raman spectra are loaded from `.mat` files. Each file corresponds to one 
donor and contains multiple spectra (typically 3 repeat measurements). The 
filename encodes the blood group label (e.g. `CAP_07_A_PP.mat` → group A) and 
the donor number (e.g. `07`). These are used to build three arrays:
- `X` — the spectra matrix (samples × wavenumbers)
- `y` — blood group labels per spectrum
- `groups` — donor ID per spectrum (used to keep repeated measurements together)

---

## Step 2: Preprocessing (done prior to this pipeline)
Before classification, each spectrum has been processed as follows:
1. **Spectral cropping** to the Raman fingerprint region (400–1800 cm⁻¹)
2. **Baseline correction** using the arPLS algorithm to remove fluorescence background
3. **Cosmic ray removal** to eliminate spike artifacts
4. **Vector normalization** to unit length to remove intensity scaling effects
5. **Mean centering** of the normalized spectra

---

## Step 3: Pairwise Classification Setup
Since SVM is inherently a binary classifier, the four-class ABO problem is 
broken down into six binary problems:

| Pair   | Class 1 | Class 2 |
|--------|---------|---------|
| AB-A   | AB      | A       |
| AB-B   | AB      | B       |
| AB-O   | AB      | O       |
| A-B    | A       | B       |
| A-O    | A       | O       |
| B-O    | B       | O       |

For each pair, only the spectra belonging to those two groups are used. Labels 
are binarized to 0 and 1.

---

## Step 4: The Classification Pipeline
Each binary classifier consists of two steps wrapped in a `sklearn` Pipeline:

### 4a. PCA Dimensionality Reduction
The raw spectra contain thousands of intensity variables (one per wavenumber). 
PCA reduces this to a small number of principal components (PCs) that capture 
the most variance. The paper uses 15 PCs as a starting point.

**Critically**, PCA is placed *inside* the pipeline so that it is refit using 
only the training data in each cross-validation fold. Fitting PCA on the full 
dataset before cross-validation would constitute data leakage.

### 4b. SVM with RBF Kernel
A Support Vector Machine with a radial basis function (RBF) kernel is used for 
classification. The RBF kernel maps the data into a higher-dimensional space 
where the two classes can be linearly separated. Two hyperparameters control 
this:
- `C` — regularization strength (how much the model penalizes misclassifications)
- `gamma` — kernel bandwidth (how far the influence of a single training point reaches)

---

## Step 5: Hyperparameter Tuning (Inner CV)
The number of PCs, `C`, and `gamma` are tuned using **5-fold group-aware 
cross-validation** inside a `GridSearchCV`. Balanced accuracy is used as the 
scoring metric rather than regular accuracy, because the dataset may be 
imbalanced (e.g. a rare antigen with few positive donors).

**Group-aware splitting** (`GroupKFold`) ensures that all spectra from the same 
donor are always kept in the same fold. This prevents the model from seeing 
repeated measurements of the same donor in both training and test, which would 
artificially inflate performance.

---

## Step 6: Evaluation (Outer CV)
The best model from Step 5 is evaluated using **5-fold group-aware 
cross-validation repeated 10 times**. This gives 50 scores in total, which are 
averaged to produce stable estimates of:
- **AUC-ROC** — the primary metric, measuring how well the classifier separates 
  the two classes across all decision thresholds. A value of 1.0 is perfect, 
  0.5 is chance level.
- **Balanced Accuracy (BA)** — the average of sensitivity and specificity, 
  robust to class imbalance.

Results are reported as **mean ± standard deviation** across the 50 folds.

---

## Expected Results
Based on Jensen et al. (2024), the target AUC values for each pair are:

| Pair | Expected AUC  |
|------|--------------|
| AB-A | 0.92 ± 0.03  |
| AB-B | 0.88 ± 0.04  |
| AB-O | 0.93 ± 0.02  |
| A-B  | 0.87 ± 0.04  |
| A-O  | 0.90 ± 0.03  |
| B-O  | 0.95 ± 0.02  |

---

## Key Design Decisions
- **PCA inside the pipeline** prevents data leakage during cross-validation
- **GroupKFold** prevents donor-level leakage across repeated measurements
- **Balanced accuracy** as the loss function handles class imbalance
- **RBF kernel** captures non-linear separability in the spectral data
- **Repeated CV** produces stable performance estimates with uncertainty bounds

In [2]:
import os
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate

In [8]:

def load_spectra(processed_dir):
    """
    Expected filename format:
    CAP_<number>_<label>_PP.mat

    Example:
    CAP_10_0_PP.mat  -> label = '0'
    CAP_07_A_PP.mat  -> label = 'A'
    """

    X_list, y_list, group_list = [], [], []
    files = sorted(f for f in os.listdir(processed_dir) if f.endswith('_PP.mat'))

    for cap_idx, fname in enumerate(files):
        parts = fname.replace('.mat', '').split('_')

        if len(parts) < 4:
            print(f"Skipping {fname} — unexpected filename format")
            continue

        donor_number = int(parts[1])
        label = parts[-2]   # the part before 'PP'

        if label not in ['A', 'B', 'AB', '0']:
            print(f"Skipping {fname} — unrecognized label '{label}'")
            continue

        fpath = os.path.join(processed_dir, fname)
        mat = sio.loadmat(fpath, simplify_cells=True)

        if "Results" not in mat:
            print(f"Skipping {fname} — no 'Results'")
            continue

        results = mat["Results"]

        if "S_corrected" not in results:
            print(f"Skipping {fname} — no 'S_corrected'")
            continue

        # shape becomes (samples, features)
        X_file = results["S_corrected"].T

        X_list.append(X_file)
        y_list.extend([label] * X_file.shape[0])
        group_list.extend([donor_number] * X_file.shape[0])

    if not X_list:
        raise ValueError("No valid data loaded.")

    X = np.vstack(X_list)
    y = np.array(y_list)
    groups = np.array(group_list)

    print(f"Loaded {X.shape[0]} spectra from {len(X_list)} donors")
    print(f"Class distribution: { {g: int((y == g).sum()) for g in np.unique(y)} }")

    return X, y, groups


processed_dir = "/Users/dons/Library/Mobile Documents/com~apple~CloudDocs/ZHAW/SS26/Track 2/Processed for SVM"
X, y, groups = load_spectra(processed_dir)

Loaded 3414 spectra from 42 donors
Class distribution: {np.str_('0'): 1707, np.str_('A'): 1707}


In [9]:
# SANITY CHECK — runs with whatever groups are available
# Once all 4 groups are loaded this will automatically run all 6 pairs
available = np.unique(y)
print(f"Groups available: {available}")

pairs_all = [('AB','A'), ('AB','B'), ('AB','0'), ('A','B'), ('A','0'), ('B','0')]

# only keep pairs where both classes are present
pairs_ready = [(c1,c2) for c1,c2 in pairs_all 
               if c1 in available and c2 in available]

print(f"Pairs ready to train: {pairs_ready}")
print(f"Pairs waiting on data: {[p for p in pairs_all if p not in pairs_ready]}")

Groups available: ['0' 'A']
Pairs ready to train: [('A', '0')]
Pairs waiting on data: [('AB', 'A'), ('AB', 'B'), ('AB', '0'), ('A', 'B'), ('B', '0')]


In [14]:
pipe = Pipeline([
    ('pca', PCA()),
    ('svm', SVC(kernel='rbf', probability=True))
])

param_grid = {
    'pca__n_components': [15],       # fix at 15 as the paper uses
    'svm__C':     [1, 10],           # just 2 values
    'svm__gamma': ['scale']          # just 1 value
}

pairs = [('AB','A'), ('AB','B'), ('AB','0'), ('A','B'), ('A','0'), ('B','0')]

results = {}

for cls1, cls2 in pairs:
    label = f"{cls1}-{cls2}"
    
    if cls1 not in np.unique(y) or cls2 not in np.unique(y):
        print(f"Skipping {label} — missing data")
        continue
    
    mask = np.isin(y, [cls1, cls2])
    X_pair   = X[mask]
    y_pair   = y[mask]
    grp_pair = groups[mask]
    y_bin    = (y_pair == cls1).astype(int)
    
    if len(np.unique(grp_pair)) < 5:
        print(f"Skipping {label} — not enough donor groups")
        continue
    
    if len(np.unique(y_bin)) < 2:
        print(f"Skipping {label} — only one class after subsetting")
        continue
    
    print(f"\nTraining: {label}")
    
    # ── this block was missing ──────────────────────────────────────────
    inner_cv = GroupKFold(n_splits=5)
    gs = GridSearchCV(
        pipe, param_grid,
        scoring='balanced_accuracy',
        cv=inner_cv,
        n_jobs=-1,
        refit=True
    )
    gs.fit(X_pair, y_bin, groups=grp_pair)
    print(f"  Best params: {gs.best_params_}")
    # ───────────────────────────────────────────────────────────────────
    
    outer_cv = GroupKFold(n_splits=5)
    aucs, bas = [], []
    for repeat in range(10):
        sc = cross_validate(
            gs.best_estimator_,
            X_pair, y_bin,
            cv=outer_cv,
            groups=grp_pair,
            scoring=['roc_auc', 'balanced_accuracy']
        )
        aucs.extend(sc['test_roc_auc'])
        bas.extend(sc['test_balanced_accuracy'])
    
    results[label] = {
        'AUC':     np.mean(aucs),
        'AUC_std': np.std(aucs),
        'BA':      np.mean(bas),
        'BA_std':  np.std(bas)
    }
    print(f"  AUC = {np.mean(aucs):.2f} ± {np.std(aucs):.2f}")
    print(f"  BA  = {np.mean(bas):.2f} ± {np.std(bas):.2f}")

Skipping AB-A — missing data
Skipping AB-B — missing data
Skipping AB-0 — missing data
Skipping A-B — missing data

Training: A-0
  Best params: {'pca__n_components': 15, 'svm__C': 1, 'svm__gamma': 'scale'}
  AUC = 0.50 ± 0.00
  BA  = 0.50 ± 0.00
Skipping B-0 — missing data


In [11]:
# Print results summary
summary = pd.DataFrame(results).T
print(summary.round(3))

Empty DataFrame
Columns: []
Index: []
